In [1]:
%pip install -q langchain langchain_openai langgraph langsmith

In [2]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "triage-capstone"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

In [3]:
%pip install -q langchain_huggingface sentence-transformers langchain-text-splitters

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

kb_texts = [
    """# Credential Access Patterns

## Brute Force / Password Guessing
Repeated failed authentication attempts against the same account or service in
a short time window, especially against privileged or default accounts
(admin, root, administrator), often from a single external IP or a small
rotating set of IPs. A high attempt count in under a minute suggests
automated tooling rather than a human mistyping a password. Severity should
scale with attempt volume, account privilege, and whether any attempt
eventually succeeded.

## Successful Login Anomalies
A successful login itself is not inherently suspicious. Context that raises
severity: new device or geography inconsistent with the user's history,
login immediately following a failed-attempt burst, or a login at an unusual
hour for that account's normal pattern.""",

    """# Malware, Persistence, and Exfiltration Patterns

## Command-and-Control (C2) Indicators
Outbound traffic to newly registered domains, or destinations already known
as C2 infrastructure. Living-off-the-land tools (powershell.exe, wscript.exe)
making outbound network connections shortly after spawning from an office
document process (winword.exe, excel.exe) is a strong indicator of a
maldoc-triggered infection chain and should be treated as high severity.

## Persistence Mechanisms
Scheduled tasks or registry run keys created with generic or disguised names
shortly after suspicious process activity. Deletion of volume shadow copies
(vssadmin delete shadows) is strongly associated with ransomware staging and
should always be treated as critical severity.

## Web Application Attacks
Classic injection patterns (SQL injection strings, path traversal) in HTTP
request parameters. A single rejected probe is low-to-medium severity; any
successful injection is high severity."""
]

docs = [Document(page_content=t) for t in kb_texts]
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

def retrieve_context(query, k=3):
    results = vector_store.similarity_search(query, k=k)
    return "\n\n".join(d.page_content for d in results)

print("Knowledge base ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Knowledge base ready.


In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [6]:
from typing import Literal
from typing_extensions import NotRequired
from langchain.agents import AgentState

class TriageState(AgentState):
    current_step: NotRequired[Literal["classifier", "resolution"]]
    threat_category: NotRequired[Literal["credential_access", "malware_c2", "web_attack", "phishing", "benign", "unknown"]]
    severity: NotRequired[Literal["low", "medium", "high", "critical"]]

In [7]:
from typing import Literal
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command, interrupt

@tool
def record_classification(
    threat_category: Literal["credential_access", "malware_c2", "web_attack", "phishing", "benign", "unknown"],
    severity: Literal["low", "medium", "high", "critical"],
    runtime: ToolRuntime[None, TriageState],
) -> Command:
    """Record the log's threat category and severity, then move to resolution."""
    return Command(
        update={
            "threat_category": threat_category,
            "severity": severity,
            "current_step": "resolution",
            "messages": [
                ToolMessage(
                    content=f"Classification recorded: {threat_category} / {severity}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

@tool
def provide_verdict(verdict: str) -> str:
    """Provide the final triage verdict/action for a low or medium severity log."""
    return f"Verdict provided: {verdict}"

@tool
def escalate_to_analyst(reason: str, summary: str, runtime: ToolRuntime) -> str:
    """Escalate this case to a human analyst. Use for high/critical severity, or when you cannot resolve it confidently."""
    payload = {
        "action": "escalate",
        "reason": reason,
        "summary": summary,
        "message": "Analyst input required.",
    }
    human_response = interrupt(payload)
    if isinstance(human_response, str):
        return human_response
    return str(human_response)

@tool
def go_back_to_classifier() -> Command:
    """Go back to the classification step to correct the category/severity."""
    return Command(update={"current_step": "classifier"})

In [8]:
CLASSIFIER_PROMPT = """You are a security analyst triage agent.

CURRENT STAGE: Classification

Reference threat patterns (retrieved for this log):
{kb_context}

At this step:
1. Read the raw log entry provided.
2. Determine its threat_category and severity, grounded in the reference patterns above.
3. Use record_classification to record it and move to the resolution step.
"""

RESOLUTION_PROMPT = """You are a security analyst triage agent.

CURRENT STAGE: Resolution
LOG INFO: threat_category is {threat_category}, severity is {severity}

At this step:
1. For LOW or MEDIUM severity: use provide_verdict to state the recommended action.
2. For HIGH or CRITICAL severity: use escalate_to_analyst to hand this off to a human.
If the classification seems wrong, use go_back_to_classifier to redo it.
"""

STEP_CONFIG = {
    "classifier": {
        "prompt": CLASSIFIER_PROMPT,
        "tools": [record_classification],
        "requires": [],
    },
    "resolution": {
        "prompt": RESOLUTION_PROMPT,
        "tools": [provide_verdict, escalate_to_analyst, go_back_to_classifier],
        "requires": ["threat_category", "severity"],
    },
}

In [9]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def apply_step_config(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    current_step = request.state.get("current_step", "classifier")
    stage_config = STEP_CONFIG[current_step]

    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    prompt_vars = dict(request.state)
    if current_step == "classifier":
        last_human = request.state["messages"][-1].content
        prompt_vars["kb_context"] = retrieve_context(last_human)

    system_prompt = stage_config["prompt"].format(**prompt_vars)

    request = request.override(
        system_prompt=system_prompt,
        tools=stage_config["tools"],
    )
    return handler(request)

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

all_tools = [record_classification, provide_verdict, escalate_to_analyst, go_back_to_classifier]

agent = create_agent(
    model=model,
    tools=all_tools,
    state_schema=TriageState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)

print("Agent ready.")

Agent ready.


In [11]:
import uuid
from langgraph.types import Command
from langchain_core.messages import HumanMessage

def triage_log_interactive():
    log_text = input("Paste a log line to triage: ")
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = agent.invoke({"messages": [HumanMessage(log_text)]}, config)

    if "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        print("\n>>> ESCALATED TO ANALYST <<<")
        print(f"Reason:  {payload.get('reason')}")
        print(f"Summary: {payload.get('summary')}")
        analyst_response = input("\nYou are the analyst. Type your decision/resolution: ")
        result = agent.invoke(Command(resume=analyst_response), config)
    else:
        print("\n>>> AUTO-RESOLVED (no escalation needed) <<<")

    print("\n--- Final agent response ---")
    print(result["messages"][-1].content)

triage_log_interactive()

Paste a log line to triage: Succesfull login

>>> AUTO-RESOLVED (no escalation needed) <<<

--- Final agent response ---
Benign low-severity successful login; no further action required.
